# E-commerce Fulfilment & Delivery Performance Analysis

## Notebook 01: Data Understanding

This notebook is the first stage of the project.

The purpose of this notebook is to:

- Load the raw Olist e-commerce CSV files
- Inspect the available tables
- Check row counts and column counts
- Review column names and data types
- Understand how the dataset can support fulfilment, delivery performance and customer satisfaction analysis

At this stage, no cleaning or modelling is being done. The focus is only on understanding the raw data.

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
# Define the raw data folder
raw_data_path = Path("../data/raw")

# Check whether the folder exists
raw_data_path.exists()

In [ ]:
# List all CSV files in the raw data folder
csv_files = list(raw_data_path.glob("*.csv"))

# Display the file names
for file in csv_files:
    print(file.name)

## Load All Raw CSV Files

In this section, I will load each raw CSV file into a pandas dataframe.

A dataframe is like an Excel table inside Python.

The aim is to understand:

- How many rows each table has
- How many columns each table has
- What each table is likely used for
- Which columns may be useful for delivery, fulfilment and customer satisfaction analysis

In [ ]:
# Load each CSV file into a pandas dataframe

customers = pd.read_csv(raw_data_path / "olist_customers_dataset.csv")
geolocation = pd.read_csv(raw_data_path / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(raw_data_path / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(raw_data_path / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(raw_data_path / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(raw_data_path / "olist_orders_dataset.csv")
products = pd.read_csv(raw_data_path / "olist_products_dataset.csv")
sellers = pd.read_csv(raw_data_path / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(raw_data_path / "product_category_name_translation.csv")

print("All CSV files loaded successfully.")

In [ ]:
# Create a dictionary to store all dataframes in one place

datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

# Create a summary table showing rows and columns for each dataset

summary = []

for name, df in datasets.items():
    summary.append({
        "table_name": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

summary_df = pd.DataFrame(summary)

summary_df

In [ ]:
# Display column names for each dataset

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("-" * 50)
    print(df.columns.tolist())

In [ ]:
# Preview the first 5 rows of each dataset

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    display(df.head())

In [ ]:
# Check data types for each dataset

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("-" * 50)
    print(df.dtypes)

## Dataset Table Overview

The Olist dataset is made up of multiple related tables.

This is useful because it allows the project to demonstrate relational data analysis, similar to how business systems store operational data across different tables.

For example:

- The orders table stores order-level information.
- The customers table stores customer location information.
- The order_items table stores product and seller information for each order.
- The reviews table stores customer satisfaction scores.
- The payments table stores payment information.
- The products table stores product details.
- The sellers table stores seller location information.

The key business value of this structure is that I can connect fulfilment performance, delivery timing, freight cost, product category, seller performance and customer review score.

In [ ]:
# Create a table explaining the purpose of each dataset

table_overview = pd.DataFrame([
    {
        "table_name": "orders",
        "purpose": "Main order-level table containing order status and key order dates.",
        "key_columns": "order_id, customer_id, order_status, order_purchase_timestamp, order_delivered_customer_date, order_estimated_delivery_date",
        "business_use": "Used to analyse delivery performance, late orders and order lifecycle timing."
    },
    {
        "table_name": "customers",
        "purpose": "Customer-level table containing customer location details.",
        "key_columns": "customer_id, customer_unique_id, customer_city, customer_state",
        "business_use": "Used to analyse delivery performance by customer geography."
    },
    {
        "table_name": "order_items",
        "purpose": "Order item-level table showing products, sellers, price and freight cost.",
        "key_columns": "order_id, order_item_id, product_id, seller_id, price, freight_value",
        "business_use": "Used to analyse freight cost, product performance and seller-level fulfilment."
    },
    {
        "table_name": "order_reviews",
        "purpose": "Customer review table containing review scores and comments.",
        "key_columns": "review_id, order_id, review_score",
        "business_use": "Used to analyse customer satisfaction and the impact of delivery delays."
    },
    {
        "table_name": "order_payments",
        "purpose": "Payment table containing payment type, instalments and payment value.",
        "key_columns": "order_id, payment_type, payment_installments, payment_value",
        "business_use": "Used to understand order value and payment behaviour."
    },
    {
        "table_name": "products",
        "purpose": "Product table containing product category and product physical attributes.",
        "key_columns": "product_id, product_category_name, product_weight_g, product_length_cm, product_height_cm, product_width_cm",
        "business_use": "Used to analyse product categories, freight cost and fulfilment complexity."
    },
    {
        "table_name": "sellers",
        "purpose": "Seller table containing seller location details.",
        "key_columns": "seller_id, seller_city, seller_state",
        "business_use": "Used to analyse seller geography and seller delivery performance."
    },
    {
        "table_name": "geolocation",
        "purpose": "Geolocation table containing postcode, latitude, longitude, city and state.",
        "key_columns": "geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state",
        "business_use": "Can support location-based analysis and mapping."
    },
    {
        "table_name": "category_translation",
        "purpose": "Translation table for product category names from Portuguese to English.",
        "key_columns": "product_category_name, product_category_name_english",
        "business_use": "Used to make product category analysis easier to understand for English-speaking stakeholders."
    }
])

table_overview

## Initial Relationship Understanding

The main table for this project is the orders table because it contains the order ID, customer ID, order status and key delivery dates.

Important relationships:

| Main Table | Connects To | Join Column | Why It Matters |
|---|---|---|---|
| orders | customers | customer_id | Adds customer location to each order |
| orders | order_items | order_id | Adds product, seller, price and freight information |
| orders | order_reviews | order_id | Adds customer review score |
| orders | order_payments | order_id | Adds payment information |
| order_items | products | product_id | Adds product category and product attributes |
| order_items | sellers | seller_id | Adds seller location |
| products | category_translation | product_category_name | Adds English product category names |

This relationship structure will allow me to create a single analysis-ready dataset later in the project.

## Initial Data Quality Checks

Before cleaning or analysing the data, I need to understand the quality of the raw files.

The purpose of these checks is to identify:

- Missing values
- Duplicate rows
- Columns that may need data type changes
- Primary keys that should be unique
- Important fields that may need further investigation

At this stage, I am only inspecting the raw data. Any cleaning decisions will be made in the next notebook.

In [ ]:
# Check missing values across all datasets

missing_summary = []

for name, df in datasets.items():
    total_cells = df.shape[0] * df.shape[1]
    total_missing = df.isna().sum().sum()
    columns_with_missing = (df.isna().sum() > 0).sum()
    missing_percentage = (total_missing / total_cells) * 100 if total_cells > 0 else 0
    
    missing_summary.append({
        "table_name": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "total_missing_values": total_missing,
        "columns_with_missing_values": columns_with_missing,
        "missing_percentage": round(missing_percentage, 2)
    })

missing_summary_df = pd.DataFrame(missing_summary)
missing_summary_df.sort_values(by="total_missing_values", ascending=False)

In [ ]:
# Show missing values by column for each dataset

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("-" * 50)
    
    missing_by_column = df.isna().sum()
    missing_by_column = missing_by_column[missing_by_column > 0].sort_values(ascending=False)
    
    if missing_by_column.empty:
        print("No missing values found.")
    else:
        print(missing_by_column)

In [ ]:
# Check duplicate rows across all datasets

duplicate_summary = []

for name, df in datasets.items():
    duplicate_rows = df.duplicated().sum()
    duplicate_percentage = (duplicate_rows / len(df)) * 100 if len(df) > 0 else 0
    
    duplicate_summary.append({
        "table_name": name,
        "rows": len(df),
        "duplicate_rows": duplicate_rows,
        "duplicate_percentage": round(duplicate_percentage, 2)
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)
duplicate_summary_df.sort_values(by="duplicate_rows", ascending=False)

In [ ]:
# Check whether key ID columns are unique

key_checks = [
    ("orders", orders, "order_id"),
    ("customers", customers, "customer_id"),
    ("products", products, "product_id"),
    ("sellers", sellers, "seller_id"),
    ("order_reviews", order_reviews, "review_id")
]

key_check_results = []

for table_name, df, key_column in key_checks:
    total_rows = len(df)
    unique_keys = df[key_column].nunique()
    duplicate_keys = total_rows - unique_keys
    
    key_check_results.append({
        "table_name": table_name,
        "key_column": key_column,
        "total_rows": total_rows,
        "unique_key_values": unique_keys,
        "duplicate_key_count": duplicate_keys,
        "is_unique": duplicate_keys == 0
    })

key_check_df = pd.DataFrame(key_check_results)
key_check_df

In [ ]:
# Check the different order statuses in the orders table

orders["order_status"].value_counts()

In [ ]:
# Check important date columns in the orders table

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns].head()

## Initial Data Quality Observations

The initial checks show that the dataset contains multiple related tables with different row counts and levels of completeness.

Key observations from this stage:

- Some tables contain missing values, especially where operational events may not have occurred or were not recorded.
- Date columns currently need further inspection and may need conversion into proper datetime format.
- Duplicate row checks help identify whether any exact repeated records exist.
- Primary key checks help confirm whether important ID columns can be used safely for joins.
- Order status values need to be reviewed before calculating delivery KPIs.

The next notebook will focus on cleaning and preparing the data for analysis.